# BrailleLens — Fingertip YOLO26 Domain Fine-Tune (Colab GPU)

Fine-tune **`yolo26n_fingertip_best.pt`** on 60 LabelMe-annotated Braille fingertip photos.

**Disconnect-safe:** checkpoints save to Google Drive every epoch.

### Before you start

1. On PC: annotate `Gold Dataset/Braille_fingertip/` → run `build_dataset.py` → `pack_for_colab.py`
2. Upload to Drive:
   - `braille_fingertip_yolo.zip` → `MyDrive/BrailleLens_Fingertip_Domain/`
   - `yolo26n_fingertip_best.pt` → `MyDrive/BrailleLens_Fingertip_Domain/`
3. **Runtime → Change runtime type → T4 GPU**
4. Run cells in order

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/BrailleLens_Fingertip_Domain")
ZIP_PATH = DRIVE_ROOT / "braille_fingertip_yolo.zip"
BASE_WEIGHTS = DRIVE_ROOT / "yolo26n_fingertip_best.pt"
RUNS_DIR = DRIVE_ROOT / "runs" / "fingertip_domain"
RUN_NAME = "yolo26n_braille_finetune"
WEIGHTS_DIR = RUNS_DIR / RUN_NAME / "weights"
LAST_PT = WEIGHTS_DIR / "last.pt"
BEST_PT = WEIGHTS_DIR / "best.pt"
METRICS_OUT = DRIVE_ROOT / "metrics_summary.json"
EXPORT_PT = DRIVE_ROOT / "yolo26n_fingertip_braille_best.pt"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("DRIVE_ROOT :", DRIVE_ROOT)
print("ZIP exists :", ZIP_PATH.exists(), "->", ZIP_PATH)
print("Weights    :", BASE_WEIGHTS.exists(), "->", BASE_WEIGHTS)

## 1) Install Ultralytics

If the runtime restarts after Pillow reinstall, re-run the Drive mount cell above, then this cell again.

In [ ]:
import os

!pip -q uninstall -y pillow
!pip -q install --no-cache-dir --force-reinstall "pillow>=11.3.0"
!pip -q install -U ultralytics pyyaml opencv-python-headless

try:
    from PIL import ImageText  # noqa: F401
    from ultralytics import YOLO  # noqa: F401
    import torch
    import ultralytics

    print("ultralytics", ultralytics.__version__)
    print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: no GPU — Runtime → Change runtime type → T4 GPU")
except ImportError as e:
    print("Import failed:", e)
    print("Restarting runtime to load new Pillow...")
    os.kill(os.getpid(), 9)

## 2) Unpack dataset to local disk (fast)

Training from mounted Drive is slow. Unpack to `/content/braille_fingertip_yolo`.

In [ ]:
import zipfile
import yaml

LOCAL_DATA = Path("/content/braille_fingertip_yolo")
DATA_YAML = LOCAL_DATA / "data.yaml"

if DATA_YAML.exists():
    print("Dataset already unpacked:", LOCAL_DATA)
else:
    if not ZIP_PATH.exists():
        raise FileNotFoundError(
            f"Missing {ZIP_PATH}\n"
            "Upload braille_fingertip_yolo.zip to MyDrive/BrailleLens_Fingertip_Domain/\n"
            "On PC: finger_cell_track/yolo_domain_specific/pack_for_colab.py"
        )
    print("Unpacking", ZIP_PATH, "-> /content ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/content")
    print("Unpacked to", LOCAL_DATA)

cfg = {
    "path": str(LOCAL_DATA.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": {0: "fingertip"},
}
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

for split in ("train", "val", "test"):
    n = len(list((LOCAL_DATA / "images" / split).glob("*.*")))
    print(f"{split:5s}: {n} images")

## 3) Optional — hyperparameter tuning (`model.tune`)

Set **`RUN_TUNE = True`** to search lr0, weight_decay, mosaic, fliplr via Ultralytics genetic algorithm.

Keep **`RUN_TUNE = False`** for a faster first run with sensible defaults.

If tuning fails on YOLO26 (known Colab issue), skip this cell and train with defaults below.

In [ ]:
from ultralytics import YOLO
import torch

RUN_TUNE = False  # set True to run ~15 tune iterations (slow)
TUNE_ITERATIONS = 15
TUNE_EPOCHS = 30

DEFAULT_HP = {
    "lr0": 0.001,
    "weight_decay": 0.0005,
    "mosaic": 0.5,
    "mixup": 0.05,
    "fliplr": 0.5,
    "dropout": 0.0,
}

best_hp = dict(DEFAULT_HP)

if RUN_TUNE:
    if not BASE_WEIGHTS.exists():
        raise FileNotFoundError(f"Upload base weights to {BASE_WEIGHTS}")
    tune_model = YOLO(str(BASE_WEIGHTS))
    search_space = {
        "lr0": (0.0003, 0.003),
        "weight_decay": (0.0001, 0.001),
        "mosaic": (0.2, 0.7),
        "fliplr": (0.0, 0.5),
    }
    try:
        tune_model.tune(
            data=str(DATA_YAML),
            epochs=TUNE_EPOCHS,
            iterations=TUNE_ITERATIONS,
            optimizer="auto",
            batch=8,
            imgsz=640,
            device=0 if torch.cuda.is_available() else "cpu",
            space=search_space,
            plots=False,
            val=True,
        )
        hp_path = Path("runs/detect/tune/best_hyperparameters.yaml")
        if hp_path.exists():
            loaded = yaml.safe_load(hp_path.read_text())
            if isinstance(loaded, dict):
                best_hp.update({k: loaded[k] for k in DEFAULT_HP if k in loaded})
            print("Loaded tuned hyperparameters:", best_hp)
        else:
            print("Tune finished but best_hyperparameters.yaml not found — using defaults")
    except Exception as exc:
        print("Tune failed — using defaults:", exc)
else:
    print("Skipping tune. Using defaults:", best_hp)

## 4) Fine-tune from `yolo26n_fingertip_best.pt`

- **Augmentation:** moderate mosaic/mixup for small Braille dataset
- **Early stopping:** `patience=15`
- **Regularization:** `weight_decay` (L2)
- **Dropout:** optional via `best_hp['dropout']` (default 0.0)
- Auto-resume from `last.pt` on Drive after disconnect

In [ ]:
from ultralytics import YOLO
import torch

EPOCHS = 80
IMGSZ = 640
BATCH = 8
PATIENCE = 15

resume = LAST_PT.exists()
print("Resume from last.pt?", resume, "|", LAST_PT)

if resume:
    model = YOLO(str(LAST_PT))
elif BASE_WEIGHTS.exists():
    model = YOLO(str(BASE_WEIGHTS))
    print("Fine-tuning from", BASE_WEIGHTS)
else:
    raise FileNotFoundError(
        f"Upload yolo26n_fingertip_best.pt to {BASE_WEIGHTS}"
    )

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    resume=resume,
    save=True,
    save_period=1,
    patience=PATIENCE,
    workers=4,
    seed=42,
    plots=True,
    # fine-tune hyperparams
    lr0=best_hp["lr0"],
    weight_decay=best_hp["weight_decay"],
    dropout=best_hp.get("dropout", 0.0),
    # augmentation (moderate for 60-image domain set)
    hsv_h=0.015,
    hsv_s=0.50,
    hsv_v=0.40,
    degrees=5.0,
    translate=0.10,
    scale=0.30,
    shear=1.0,
    perspective=0.0005,
    flipud=0.0,
    fliplr=best_hp["fliplr"],
    mosaic=best_hp["mosaic"],
    mixup=best_hp["mixup"],
    close_mosaic=10,
)

print("\nDone / paused.")
print("best.pt ->", BEST_PT, "exists:", BEST_PT.exists())
print("last.pt ->", LAST_PT, "exists:", LAST_PT.exists())

## 5) Evaluate — Precision, Recall, F1, mAP50

Runs validation on **val** and **test** splits. Saves `metrics_summary.json` to Drive.

In [ ]:
import json
import shutil
from ultralytics import YOLO
from IPython.display import Image, display

if not BEST_PT.exists():
    raise FileNotFoundError(f"No best.pt at {BEST_PT} — finish training first")

model = YOLO(str(BEST_PT))


def eval_split(split: str) -> dict:
    m = model.val(
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        device=0 if torch.cuda.is_available() else "cpu",
        conf=0.25,
        split=split,
        plots=(split == "val"),
    )
    p = float(m.box.mp)
    r = float(m.box.mr)
    f1 = 2 * p * r / (p + r + 1e-9)
    return {
        "precision": round(p, 4),
        "recall": round(r, 4),
        "f1": round(f1, 4),
        "map50": round(float(m.box.map50), 4),
        "map50_95": round(float(m.box.map), 4),
    }


summary = {
    "base_weights": str(BASE_WEIGHTS.name),
    "best_weights": str(BEST_PT),
    "val": eval_split("val"),
    "test": eval_split("test"),
    "hyperparameters": best_hp,
}

METRICS_OUT.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("\nSaved:", METRICS_OUT)

for plot_name in ("results.png", "confusion_matrix.png", "val_batch0_pred.jpg"):
    p = RUNS_DIR / RUN_NAME / plot_name
    if p.exists():
        display(Image(filename=str(p)))

## 6) Export weights to Drive

Download `yolo26n_fingertip_braille_best.pt` and place on PC at:
`finger_cell_track/weights/yolo26n_fingertip_braille_best.pt`

In [ ]:
import shutil

if not BEST_PT.exists():
    raise FileNotFoundError(f"Missing {BEST_PT}")

shutil.copy2(BEST_PT, EXPORT_PT)
print("Exported:", EXPORT_PT)
print("Also copy metrics:", METRICS_OUT)
print("\nPaste metrics into finger_cell_track/yolo_domain_specific/metrics_template.md")